<a href="https://colab.research.google.com/github/Abdoulaye279/Financial-Engineering/blob/main/lehman-ai-agents/part1_full_context.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lehman 10-Q (July 2008): the whole filing in one prompt

Asking Gemini for the warning signs in Lehman's last 10-Q before the bankruptcy, each one with an exact quote, then checking the quotes. RAG version in `part2_rag.ipynb`.

Gemini key: Colab secret `GEMINI_API_KEY`, or env variable `GOOGLE_API_KEY` locally. Filing: [SEC text](https://www.sec.gov/Archives/edgar/data/806085/000110465908045115/a08-18147_110q.htm)

In [2]:
!pip install -q edgartools==5.21.1 langchain==1.2.10 langchain-google-genai==4.2.1 pydantic==2.12.3 rich==13.9.4

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 1.75.0 which is incompatible.


In [4]:
import os
import time
from typing import List
from pydantic import BaseModel, Field
from edgar import Company, set_identity
from google import genai
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich import box
from rich.text import Text

LEHMAN_CIK = "806085"
set_identity("abdoullayekoita897@gmail.com")  # SEC requires an email to download filings

# Gemini API key: Colab secret if available, otherwise the GOOGLE_API_KEY environment variable
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")
except ImportError:
    assert "GOOGLE_API_KEY" in os.environ, "set GOOGLE_API_KEY before running"

console = Console()
console.print("Libraries imported and environment configured.")

Libraries imported and environment configured.

In [5]:
def fetch_lehman_filings(start_date="2008-07-01"):
    lehman = Company(LEHMAN_CIK)
    target = []
    for form in ["10-K", "10-Q"]:
        for f in lehman.get_filings(form=form):
            if str(f.filing_date) >= start_date:
                target.append(f)
    return target

filings = fetch_lehman_filings()
console.print(f"Found {len(filings)} filings:")
for f in filings:
    console.print(f"  - {f.form} filed on {f.filing_date}")
# only one filing comes back: the 10-Q of 10 july 2008 (no 10-K after july 2008)

Found 1 filings:

- 10-Q filed on 2008-07-10

In [6]:
model = "gemini-3-flash-preview"   # switch to "gemini-2.5-flash" if the quota is exceeded

parts = []
for f in filings:
    console.print(f"Extracting text from {f.form} ({f.filing_date})...")
    parts.append(f"=== {f.form} filed {f.filing_date} ===\n{f.text()}")
full_context = "\n\n".join(parts)

genai_client = genai.Client()
n_tokens = genai_client.models.count_tokens(model=model, contents=full_context).total_tokens
console.print(f"Full context: {len(full_context):,} characters, {n_tokens:,} tokens")
# 535k characters, ~110k tokens -> fits in gemini's context, but every question resends all of it

Extracting text from 10-Q (2008-07-10)...

Full context: 535,654 characters, 109,582 tokens

In [7]:
class BankruptcyWarningSign(BaseModel):
    warning_type: str = Field(description="Type of warning (e.g., Leverage, Exposure, Capital, Liquidity)")
    key_metric: str = Field(description="Specific financial metric or event")
    value: str = Field(description="Quantified value if available")
    severity: int = Field(description="Severity score 1-10", ge=1, le=10)
    description: str = Field(description="Detailed explanation of the concern")
    exact_passage: str = Field(description="Exact quote from the filing, copied word for word")

class InvestigationReport(BaseModel):
    findings: List[BankruptcyWarningSign]
    summary: str


llm = ChatGoogleGenerativeAI(model=model, temperature=0)
structured_llm = llm.with_structured_output(InvestigationReport)

prompt = ChatPromptTemplate.from_template("""
You are a financial forensic analyst. Based ONLY on the following Lehman Brothers 2008 filings,
identify specific bankruptcy warning signs. Look in particular at:
- leverage ratios and debt levels (gross and net leverage)
- exposure to subprime, residential and commercial mortgage-backed securities
- capital raises, liquidity and funding concerns
- significant losses, especially in the capital markets segment
- Repo 105 or other off-balance sheet treatments used to manage leverage

For each warning sign, copy the supporting passage word for word from the filings.

Filings:
{context}
""")

start = time.time()
report = (prompt | structured_llm).invoke({"context": full_context})
console.print(f"Answer received in {time.time() - start:.0f} s")
# ~30 s for the answer

Answer received in 30 s

In [8]:
def display_report(report):
    console.print(Panel(Text("LEHMAN BROTHERS BANKRUPTCY INVESTIGATION REPORT (FULL CONTEXT)", style="bold white on red"), box=box.DOUBLE))

    table = Table(title="Detected Warning Signs", box=box.ROUNDED)
    table.add_column("Type", style="cyan")
    table.add_column("Metric", style="magenta")
    table.add_column("Value", style="green")
    table.add_column("Sev", justify="center")
    table.add_column("Description", style="white")
    for f in report.findings:
        color = "red" if f.severity >= 8 else "yellow" if f.severity >= 5 else "green"
        table.add_row(f.warning_type, f.key_metric, f.value, f"[{color}]{f.severity}[/{color}]", f.description)

    console.print(table)
    console.print(Panel(report.summary, title="Executive Summary", border_style="blue"))
    console.print("\n[bold]Supporting Passages:[/bold]")
    for i, f in enumerate(report.findings, 1):
        console.print(f"\n{i}. [italic]'{f.exact_passage}'[/italic]")

display_report(report)
# 10 signs: $2.8bn loss, $4bn write-downs, $3.1bn collateral calls, negative capital markets revenues,
# $37.9bn level 3 assets, total assets -19% in a quarter, $4bn share issue in june

╔═════════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
║ LEHMAN BROTHERS BANKRUPTCY INVESTIGATION REPORT (FULL CONTEXT)                                                  ║
╚═════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝

                                              Detected Warning Signs                                               
╭───────────┬───────────────────────────────────┬───────────────────────┬─────┬───────────────────────────────────╮
│ Type      │ Metric                            │ Value                 │ Sev │ Description                       │
├───────────┼───────────────────────────────────┼───────────────────────┼─────┼───────────────────────────────────┤
│ Capital   │ Net Income/Loss                   │ $2.8 billion loss     │ 10  │ The firm reported a massive net   │
│           │                                   │                       │     │ loss for the second quarter of    │
│           │                                   │                       │     │ 2008, a sharp reversal from the   │
│           │                                   │                       │     │ prior year's profit.              │
│ Exposure  │ Negative Valuation Adjustments    │ $4.0 billion          │ 10  │ Significant write-downs on        │
│           │                                   │                       │     │ financial inventory due to market │
│           │                                   │                       │     │ dislocation and asset repricing.  │
│ Liquidity │ Collateral Requirements           │ $3.1 billion          │ 10  │ Counterparties gained the right   │
│           │                                   │                       │     │ to demand billions in additional  │
│           │                                   │                       │     │ collateral, creating a direct     │
│           │                                   │                       │     │ threat to liquidity.              │
│ Capital   │ Capital Markets Net Revenues      │ Negative $2.4 billion │ 10  │ The core Capital Markets segment  │
│           │                                   │                       │     │ experienced negative net          │
│           │                                   │                       │     │ revenues, indicating a failure in │
│           │                                   │                       │     │ primary trading and market-making │
│           │                                   │                       │     │ activities.                       │
│ Exposure  │ U.S. Subprime Residential         │ $2,755 million        │  9  │ Continued exposure to toxic       │
│           │ Mortgages                         │                       │     │ subprime mortgage assets despite  │
│           │                                   │                       │     │ market deterioration.             │
│ Exposure  │ Commercial Mortgage-Related       │ $29.4 billion         │  9  │ Large concentration of illiquid   │
│           │ Assets                            │                       │     │ commercial mortgage-backed        │
│           │                                   │                       │     │ securities and whole loans.       │
│ Liquidity │ Level 3 Assets                    │ $37.9 billion         │  8  │ A significant portion of the      │
│           │                                   │                       │     │ balance sheet consists of         │
│           │                                   │                       │     │ illiquid assets where fair value  │
│           │                                   │                       │     │ is based on unobservable inputs.  │
│ Capital   │ Post-Quarter Capital Raise        │ $6.0 billion          │  9  │ The firm was forced to raise $4   │
│           │                                   │                       │     │ billion in common stock and $2    │
│           │                                   │                       │     │ billion in preferred stock        │
│           │                                   │                       │     │ immediately after the quarter     │
│           │                                   │       

╭─────────────────────────────────────────────── Executive Summary ───────────────────────────────────────────────╮
│ Lehman Brothers' Q2 2008 filing depicts a firm in a critical liquidity and capital crisis. The firm reported a  │
│ $2.8 billion net loss, driven by $4.0 billion in negative valuation adjustments on mortgage-related assets and  │
│ negative net revenues in its core Capital Markets segment. Despite a massive 19% reduction in total assets      │
│ ($146.6 billion) to manage leverage, the firm remained exposed to $37.9 billion in illiquid Level 3 assets and  │
│ faced $3.1 billion in potential collateral calls. The situation necessitated an emergency $6 billion capital    │
│ raise on June 12, 2008, shortly after the quarter's close, as credit spreads widened significantly, signaling   │
│ high default risk.                                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Supporting Passages:

1. 'The Company reported a net loss for the second quarter of 2008 of $2.8 billion, or a $5.14 loss per common 
share (diluted), compared with net earnings for the second quarter of 2007 of $1.3 billion, or $2.21 per common 
share (diluted).'

2. 'As a result of these market events, the Company recorded negative valuation adjustments of approximately $4.0 
billion on certain components of its financial inventory.'

... (4 outputs left out)

7. 'At May 31, 2008, Level 3 assets, including derivatives on a net basis, were approximately $37.9 billion, a 6% 
decrease from approximately $40.2 billion at the end of the 2008 first quarter and a 3% decline from approximately 
$38.9 billion at the end of the 2007 fiscal year.'

8. 'On June 12, 2008, the Company completed the issuance and sale of 143 million shares of its common stock, par 
value $0.10 (“ Common Stock”). The sale of the Common Stock closed at a price of $28.00 per share, resulting in 
approximately $4.0 billion of proceeds for the Company.'

9. 'At May 31, 2008, the Company’s total assets were approximately $639.4 billion, representing a 19% decrease, or 
a reduction of $146.6 billion, from total asset levels of $786.0 billion at February 29, 2008.'

10. 'At May 31, 2008, the estimated changes in the fair value attributable to the observable impact from 
instrument-specific credit risk was a gain of approximately $0.4 billion and $1.0 billion for the 2008 three and 
six months ended, respectively, attributable to the widening of the Company’s credit spreads during these periods.'

In [9]:
import re

def normalize(s):
    return re.sub(r"\s+", " ", s).strip().lower()

def check_quotes(report, source_text):
    """Check that every quoted passage really appears in the text the model was given."""
    src = normalize(source_text)
    ok = 0
    for i, f in enumerate(report.findings, 1):
        found = normalize(f.exact_passage) in src
        ok += found
        console.print(f"{i}. {'[green]found[/green]' if found else '[red]NOT FOUND[/red]'}  {f.key_metric}")
    console.print(f"\n{ok}/{len(report.findings)} quotes found word for word in the source text")
    return ok, len(report.findings)

check_quotes(report, full_context)
# 7/10 word for word. 5 and 6 rebuild table rows with "...", 7 changes the start
# ("at may 31, 2008" instead of "at the end of the second quarter of 2008")
# checked the 3 by hand in the PDF: the numbers are all in the filing, only the wording around them is off

1. found  Net Income/Loss

2. found  Negative Valuation Adjustments

3. found  Collateral Requirements

4. found  Capital Markets Net Revenues

5. NOT FOUND  U.S. Subprime Residential Mortgages

6. NOT FOUND  Commercial Mortgage-Related Assets

... (2 outputs left out)

9. found  Total Asset Reduction

10. found  Credit Spread Widening

7/10 quotes found word for word in the source text

(7, 10)